#### Historical and Generated Attributes will be Combined Here

##### Imports

In [ ]:
from pathlib import Path
import pandas as pd

##### Directories and File Locations

In [ ]:
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
GENERATED_DIR = BASE_DIR / "data" / "generated"

ATTRIBUTES_PATH = PROCESSED_DIR / "all_attributes.csv"
HISTORICAL_GENERATED_PATH = GENERATED_DIR / "historical_generated_attributes.csv"
OUTPUT_PATH = PROCESSED_DIR / "master_attributes.csv"

##### Load Attributes

In [ ]:
attributes_df = pd.read_csv(ATTRIBUTES_PATH)
historical_generated_df = pd.read_csv(HISTORICAL_GENERATED_PATH)

##### Validate Headings

In [ ]:
print(f"Attributes DataFrame columns: {attributes_df.columns}")
print(f"Historical Generated DataFrame columns: {historical_generated_df.columns}")

##### Normalize Team Names

In [ ]:
team_name_map = {
    "ATL": "Atlanta Hawks",
    "BKN": "Brooklyn Nets",
    "BOS": "Boston Celtics",
    "CHA": "Charlotte Hornets",
    "CHI": "Chicago Bulls",
    "CLE": "Cleveland Cavaliers",
    "DAL": "Dallas Mavericks",
    "DEN": "Denver Nuggets",
    "DET": "Detroit Pistons",
    "GSW": "Golden State Warriors",
    "HOU": "Houston Rockets",
    "IND": "Indiana Pacers",
    "LAC": "Los Angeles Clippers",
    "LAL": "Los Angeles Lakers",
    "MEM": "Memphis Grizzlies",
    "MIA": "Miami Heat",
    "MIL": "Milwaukee Bucks",
    "MIN": "Minnesota Timberwolves",
    "NJN": "New Jersey Nets",
    "NOH": "New Orleans Hornets",
    "NOP": "New Orleans Pelicans",
    "NYK": "New York Knicks",
    "OKC": "Oklahoma City Thunder",
    "ORL": "Orlando Magic",
    "PHI": "Philadelphia 76ers",
    "PHX": "Phoenix Suns",
    "POR": "Portland Trail Blazers",
    "SAC": "Sacramento Kings",
    "SAS": "San Antonio Spurs",
    "TOR": "Toronto Raptors",
    "UTA": "Utah Jazz",
    "WAS": "Washington Wizards"
}

historical_generated_df = historical_generated_df.rename(
    columns={
        "TEAM_ABBREVIATION": "team"
    }
)

historical_generated_df["team"] = historical_generated_df["team"].map(team_name_map)

##### Validate Team Mapping

In [ ]:
missing_teams = historical_generated_df[historical_generated_df["team"].isnull()]

if not missing_teams.empty:
    raise ValueError(
        f"Missing team mappings: {missing_teams['TEAM_ABBREVIATION'].unique().tolist()}"
    )

##### Combine Datasets

In [ ]:
column_order = [
    "name",
    "season",
    "team"
] + [
    col for col in attributes_df.columns
    if col not in ["name", "season", "team"]
]

attributes_df = attributes_df[column_order]
historical_generated_df = historical_generated_df[column_order]

master_df = pd.concat(
    [
        historical_generated_df,
        attributes_df
    ],
    ignore_index=True
)

##### Sort Master Dataset

In [ ]:
master_df = master_df.sort_values(
    by=["season", "team", "name"]
).reset_index(drop=True)

master_df = master_df[column_order]

master_df.head()

##### Validate the Combine

In [ ]:
print(f"Attributes DataFrame shape: {attributes_df.shape}")
print(f"Historical Generated DataFrame shape: {historical_generated_df.shape}")
print(f"Master DataFrame shape: {master_df.shape}")

missing = master_df.isnull().sum()

print(missing[missing > 0])

##### Save Completed Dataset

In [ ]:
master_df.to_csv(
    OUTPUT_PATH,
    index=False
)

OUTPUT_PATH